In [9]:
# Search Parameter Tuning - вместо того, чтобы гадать какой из 
# вариантов лучше мы можем использовать 

In [10]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

for doc in documents:
  if doc["course"] == "llm-zoomcamp":
    documents_llm.append(doc)

documents = documents_llm
index = build_index(documents) 

In [11]:
def text_search(query):
  boost_dict = {"question": 3.0, "section": 0.5}

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict
  )

In [12]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [13]:
df_ground_truth.head()

,question,course,document
0,Can I take this course at my own pace and stil...,llm-zoomcamp,69d122f12e
1,Is a certificate available if I complete the c...,llm-zoomcamp,69d122f12e
2,Do self-paced learners get any certificate for...,llm-zoomcamp,69d122f12e
3,Why are certificates not issued for the self-p...,llm-zoomcamp,69d122f12e
4,Is peer review of capstone projects required i...,llm-zoomcamp,69d122f12e


In [14]:
q = ground_truth[0]
q

'''
{
  'question': 'I found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'
}
'''

"\n{\n  'question': 'I found this course late — can I still enroll and follow along?',\n  'document': '74eb249bbf'\n}\n"

In [15]:
doc_id = q["document"] 
doc_query = q["question"] 

results = text_search(query = doc_query)

In [16]:
for d in results:
  print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

69d122f12e == 69d122f12e: True
74eb249bbf == 69d122f12e: False
9f689c185f == 69d122f12e: False
5cc511f85b == 69d122f12e: False
977bf7786c == 69d122f12e: False


In [17]:
relevance = []

for d in results:
  relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [18]:
def compute_relevance_text(q):
  doc_id = q["document"]

  results = text_search(query = doc_query)

  relevance = []
  for d in results:
    relevance.append(int(d["id"] == doc_id))

  return relevance

q = ground_truth[0]
print(doc_query)
compute_relevance_text(q)

Can I take this course at my own pace and still receive a certificate at the end?


[1, 0, 0, 0, 0]

In [19]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
  relevance_total = []

  for q in tqdm(ground_truth):
    relevance = compute_relevance_text(q)
    relevance_total.append(relevance)

  return relevance_total

ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [20]:
def compute_relevance(q, search_function):
  doc_id = q["document"]
  results = search_function(query = doc_query)

  relevance = []
  for d in results:
    relevance.append(int(d["id"] == doc_id))

  return relevance


In [21]:
def compute_relevance_total(ground_truth, search_function):
  relevance_total = []

  for q in tqdm(ground_truth):
    relevance = compute_relevance(q, search_function)
    relevance_total.append(relevance)

  return relevance_total

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [22]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

  0%|          | 0/395 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0,

In [23]:
sample = [
  [1, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [0, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [0, 0, 1, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
]

In [24]:
def hit_rate(relevance):
  cnt = 0

  for line in relevance:
    if 1 in line:
      cnt = cnt + 1

  return cnt / len(relevance)

hit_rate(sample)          # 0.9333333333333333

0.9333333333333333

In [25]:
def mrr(relevance):
  total_score = 0.0

  # line - это 1 массив
  for line in relevance:
    for rank in range(len(line)):
      if line[rank] == 1:
        formula = 1 / (rank + 1)
        total_score = total_score + formula
        break

  return total_score / len(relevance)

mrr(sample)     # 0.822

0.8222222222222222

In [26]:
def evaluate(ground_truth, search_function):
  relevance_total = compute_relevance_total(ground_truth, search_function)

  return {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total),
  }

evaluate(ground_truth, text_search)


  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.05063291139240506, 'mrr': 0.025738396624472578}

In [27]:
# ! ===================================
# ! ===================================
print("==== Начало Search Parameter Tuning ====")
# ! ===================================
# =====================================

==== Начало Search Parameter Tuning ====


In [39]:
# Логика такова - если пользовать задает вопрос и если этот вопрос
# совпадает с набором данных FAQ, то это лучше чем если бы он совпадал 
# с какими-то случайно словами в ответе 
def text_search_v2(query):
  boost_dict = {"question": 2.0, "section": 0.5}

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict
  )

In [ ]:
evaluate(ground_truth, text_search_v2)

# {'hit_rate': 0.0379746835443038, 'mrr': 0.023206751054852325}

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.0379746835443038, 'mrr': 0.023206751054852325}

In [ ]:
# Функция поиск повышения рейтинга
def search_boost(query, question_boost):
  boost_dict = {"question": question_boost, "section": 0.5}

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict,
  )

In [ ]:
# Оценим несколько значений ускорения

for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,

        lambda query, 
               boost = boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

'''
  boost = 0.5: 
    { 'hit_rate': 0.9113924050632911, 'mrr': 0.800548523206751}
  boost = 1.0: 
    { 'hit_rate': 0.9240506329113924, 'mrr': 0.8139240506329113}
  boost = 3.0: 
    { 'hit_rate': 0.8987341772151899, 'mrr': 0.7693248945147676}
  boost = 5.0: 
    { 'hit_rate': 0.8708860759493671, 'mrr': 0.7401265822784809}
  boost = 10.0: 
    { 'hit_rate': 0.8582278481012658, 'mrr': 0.7122362869198313}
'''

In [ ]:
# Когда увеличиваем boost на 0.5 - это означает, что ответ в 2 раза важнее

In [ ]:
# Теперь мы можем проверить все - у нас есть увеличения вопросов, ответов и разделов
# данный поиск называется поиск по сетке. Мы таким образом можем посмотреть разные
# комбинации и решить какая лучшая для нас
def search_boosts(query, question_boost, answer_boost, section_boost):
  boost_dict = {
    "question": question_boost,
    "section": section_boost,
    "answer": answer_boost,
  }

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict,
  )

In [ ]:
results = []

# Просматриваем увеличение вопросов
for question_boost in [1.0, 2.0, 5.0]:
  # Просматриваем увеличение ответов
  for answer_boost in [1.0, 2.0, 4.0, 10.0]:
    # Просматриваем увеличение разделов
    for section_boost in [0.1, 0.2, 0.5]:
      print(
        f"Evaluating question_boost={question_boost},"
        f" answer_boost={answer_boost},"
        f" section_boost={section_boost}..."
      )

      result = evaluate(
        ground_truth,
        lambda 
          query, 
          question_boost = question_boost, 
          answer_boost = answer_boost, 
          section_boost = section_boost: search_boosts(
            query,
            question_boost,
            answer_boost,
            section_boost
          )
        )

      results.append({
        "question": question_boost,
        "answer": answer_boost,
        "section": section_boost,
        "hit_rate": result["hit_rate"],
        "mrr": result["mrr"],
      })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [41]:
# Помещаем в DataFrame, чтобы показать результат сверху

df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(20)

,question,answer,section,hit_rate,mrr
26,5.0,1.0,0.5,0.050633,0.025738
29,5.0,2.0,0.5,0.050633,0.025738
14,2.0,1.0,0.5,0.037975,0.023207
13,2.0,1.0,0.2,0.037975,0.023207
25,5.0,1.0,0.2,0.037975,0.023207
28,5.0,2.0,0.2,0.037975,0.023207
30,5.0,4.0,0.1,0.037975,0.023207
32,5.0,4.0,0.5,0.037975,0.023207
27,5.0,2.0,0.1,0.037975,0.023207
24,5.0,1.0,0.1,0.037975,0.023207


In [ ]:
# Определить функцию поиска с использованием следующих
# параметров усиления:
def text_search(query):
  boost_dict = {
    "question": 1.0,
    "answer": 2.0,
    "section": 0.1,
  }

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict,
  )